# RUNG 12P / Part A — can a passive death wave be CONTAINED? (CPU, no GPU)

**Anshuman's propagation idea:** seed apoptosis in a few cancer cells and let it SPREAD cell-to-cell — decoupling killing from per-cell recognition (the wall R5–R11 mapped). A *passive* death wave travels through **gap junctions (connexins)** — the established bystander-effect channel.

**The make-or-break question, asked of the atlas:** is there a connexin worst-donor **VITAL-LOW across ALL vital tissues** (a containable channel) that is also **tumour-expressed** (usable)?

- **No connexin vital-silent everywhere → every channel leaks into heart/liver/brain → a passive wave CANNOT be contained → the relay must be RECOGNITION-GATED per hop** (decisive; sets up Part B).
- **A vital-silent + tumour-expressed connexin exists → candidate passive channel** (surprise +) → Part B percolation sim.

**CPU runtime** (no GPU). Resumable Drive tiles + heartbeat. Handles the RUNG-8 unmeasured-gene trap with a **housekeeping depth control** (a connexin counts as 'off' only in deep cells). Reuses your RUNG-5 tumour cache for the tumour side — same Google account.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Drive (connexin tiles + reuse RUNG-5 tumour cache) + run log
import os, os.path as p
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd = '/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG12P_CACHE'] = f'{cd}/rung12p_tiles'; os.makedirs(os.environ['RUNG12P_CACHE'], exist_ok=True)
    os.environ['LOGICGATE_CACHE'] = f'{cd}/rung5_cache.npz'   # tumour connexin coverage (from the surfaceome panel)
    has = p.exists(f'{cd}/rung5_cache.r5.tumour.npz')
    print('[CELL 2] Drive mounted. RUNG-5 tumour cache:', 'FOUND' if has else 'NOT FOUND -> vital screen still decisive')
except Exception as e:
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — tiles in /content (lost on disconnect)')
from scripts.runlog import new_log
RUNLOG = new_log('rung12p_connexin', repo_dir='.')

In [ ]:
#@title Cell 3 — VALIDATE the screen (CPU, synthetic, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/34_connexin_containment.py','selftest'], RUNLOG)
print('[CELL 3]', '✓ validated' if rc==0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install CELLxGENE Census
import sys, importlib.util
from scripts.runlog import run_logged
for pk,pn in [('cellxgene_census','cellxgene-census'),('scanpy','scanpy')]:
    if importlib.util.find_spec(pk) is None:
        run_logged([sys.executable,'-m','pip','install','-q',pn], RUNLOG, label=f'pip {pn}')
print('[CELL 4] ✓ (restart if asked, then Run all — tiles make resume instant)')

In [ ]:
#@title Cell 5 — REAL run: fetch connexin vital expression + containment screen (RESUMABLE)
#@markdown CPU, ~27 connexin/HK genes x vital cells. [heartbeat] every ~20s; [N/9] tissue progress.
#@markdown Disconnect-safe: re-run this cell, Drive tiles are skipped. Watch the DECISIVE line.
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/34_connexin_containment.py','run'], RUNLOG)
print('[CELL 5]', '✓ done' if rc==0 else f'✗ exit {rc} — re-run to resume from last cached tissue')
from IPython.display import Image, display
if os.path.exists('runs/rung12p_connexin/rung12p_connexin.png'): display(Image('runs/rung12p_connexin/rung12p_connexin.png'))
if os.path.exists('runs/rung12p_connexin/rung12p_connexin.json'):
    d=json.load(open('runs/rung12p_connexin/rung12p_connexin.json')); print('DECISIVE:', d['DECISIVE'])

In [ ]:
#@title Cell 6 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung12p_connexin_','').replace('.log','')
b=f'/content/rung12p_run_{stem}.zip'
ps=sorted(set(glob.glob('runs/rung12p_connexin/*.png')+glob.glob('runs/rung12p_connexin/*.json')+[str(RUNLOG)]))
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 6] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')